# 00 · Data preparation (CMU-MOSEI)

**Run once.** Converts MMSA's `aligned_50.pkl` (4.7 GB) into `mosei_compact.npz` (about 0.55 GB), which every other
notebook loads in seconds.

1. Turn on *Internet*. Paste the Google Drive link of `aligned_50.pkl` into `GDRIVE_URL` (or attach the pkl).
2. Run all. Then *Save Version* → open the version's *Output* → *New Dataset* from `mosei_compact.npz`.
3. Attach that dataset to notebooks 01–07.

| | |
| --- | --- |
| Labels used | 1: sentiment score in [−3, 3] (the MMSA file has no emotion labels) |
| Splits | 16,326 train / 1,871 validation / 4,659 test |
| Text | BERT 50 × 768 → PCA to 128 dims (fitted on train) |
| Audio | COVAREP 50 × 74, standardised with train statistics |
| Video | FACET 50 × 35, standardised with train statistics |

In [ ]:
# Setup: make the `fusionlab` library importable. On Kaggle, always use a FRESH clone of the latest code (kept in /tmp,
# so it is not saved into this notebook's output); attached copies are used only when there is no Internet.
import glob, os, subprocess, sys

REPO_URL = 'https://github.com/fasihh/bigthing'      # the JF/ folder holds this project
BRANCH = 'jf-experiments'                           # used only until JF/ is merged into main


def fresh_clone(dest='/tmp/bigthing'):
    if not os.path.isdir(os.path.join(dest, '.git')):
        r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, dest], capture_output=True, text=True)
        if r.returncode:
            return None
    if not os.path.isdir(os.path.join(dest, 'JF')):  # JF not merged into main yet: use its branch
        subprocess.run(['git', '-C', dest, 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', dest, 'checkout', '-q', 'FETCH_HEAD'], check=True)
    return os.path.join(dest, 'JF')


def find_jf():
    here = os.path.abspath('.')
    if not here.startswith('/kaggle'):                # running inside a local checkout of the repo
        for c in (os.path.dirname(here), here, os.path.join(here, 'JF')):
            if os.path.isdir(os.path.join(c, 'src', 'fusionlab')):
                return c
    jf = fresh_clone()
    if jf:
        return jf
    for c in glob.glob('/kaggle/input/**/JF', recursive=True):   # offline fallback (may be older code)
        if os.path.isdir(os.path.join(c, 'src', 'fusionlab')):
            print('WARNING: no Internet, using an attached copy of the code:', c)
            return c
    raise RuntimeError('Cannot find the fusionlab code: turn on Internet in the notebook settings.')


JF = find_jf()
sys.path.insert(0, os.path.join(JF, 'src'))
CONFIGS = os.path.join(JF, 'configs')
commit = subprocess.run(['git', '-C', JF, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('using fusionlab from', JF, '| commit', commit or 'n/a')

In [ ]:
from fusionlab import config, data
import numpy as np

GDRIVE_URL = ''          # link to the FILE aligned_50.pkl, or leave empty if the pkl / compact file is attached
cfg = config.load_config(os.path.join(CONFIGS, '00_data_prep.yaml'))
D = data.load_mosei(cfg, GDRIVE_URL)

## Numbers to report

In [ ]:
import pandas as pd
rows = []
for s in ('train', 'valid', 'test'):
    y = D[s]['y'].numpy()
    rows.append(dict(split=s, clips=len(y), label_mean=y.mean(), label_std=y.std(), positive=(y > 0).mean(),
                     neutral=(y == 0).mean(), negative=(y < 0).mean(),
                     text_tokens=D[s]['tm'].float().sum(1).mean().item(),
                     audio_frames=D[s]['am'].float().sum(1).mean().item(),
                     video_frames=D[s]['vm'].float().sum(1).mean().item()))
stats_df = pd.DataFrame(rows).round(3)
os.makedirs(os.path.join(cfg['out_dir'], 'tables'), exist_ok=True)
stats_df.to_csv(os.path.join(cfg['out_dir'], 'tables', 'dataset_stats.csv'), index=False)
stats_df